# 🎯 Phase 1.5. Severity Engine Improvement Sprint

**Goal:**  
Improve the Severity Engine from *macro-F1 = 0.16* (near-random) to a more
respectable *0.45–0.60* by addressing the weaknesses revealed in validation.

This sprint does **not** repeat the project.  
It builds on the infrastructure already created:

- Centralized scoring engine (`src/analysis.py`)
- Stratified 200-review sample
- Human labeling tool
- Validation notebook
- Dashboard skeleton

We are simply **fixing what validation exposed**.


In [45]:
import os
from pathlib import Path

# Check current working directory
print(f"Current directory: {os.getcwd()}")

# Check if file exists
labels_path = Path("../data/labels/labels.csv")
print(f"\nLooking for: {labels_path.absolute()}")
print(f"File exists: {labels_path.exists()}")

# List what's in data/labels/
labels_dir = Path("data/labels")
if labels_dir.exists():
    print(f"\nFiles in data/labels/:")
    for f in labels_dir.iterdir():
        print(f"  - {f.name}")

Current directory: c:\Users\kirim\Downloads\FINTECH PROJECT 1\Fintech-Sentiment-Intelligence-Analysis\notebooks

Looking for: c:\Users\kirim\Downloads\FINTECH PROJECT 1\Fintech-Sentiment-Intelligence-Analysis\notebooks\..\data\labels\labels.csv
File exists: True


# Error Analysis: Why Severity Engine Failed

**Goal**: Analyze the 200 labeled reviews to understand exactly what the model got wrong.

This notebook will:
1. Load your manual labels
2. Compare them to model predictions
3. Identify patterns in misclassifications
4. Generate actionable improvement priorities

In [46]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load your manual labels (just review_id + true_sentiment + true_severity)
labels_path = Path("../data/labels/labels.csv")
labels_df = pd.read_csv(labels_path)

# Load the original data with review text and model predictions
original_path = Path("../data/labels/label_queue.csv")
original_df = pd.read_csv(original_path)

# Merge them together on review_id
df = original_df.merge(
    labels_df[['review_id', 'true_sentiment', 'true_severity', 'notes']], 
    on='review_id', 
    how='left',
    suffixes=('', '_new')
)

# Use the new labels, overwriting any old ones
if 'true_sentiment_new' in df.columns:
    df['true_sentiment'] = df['true_sentiment_new']
    df['true_severity'] = df['true_severity_new']
    df.drop(['true_sentiment_new', 'true_severity_new'], axis=1, inplace=True)

print(f"Loaded {len(df)} labeled reviews")
print(f"\nLabels present:")
print(f"  true_sentiment: {df['true_sentiment'].notna().sum()} labeled")
print(f"  true_severity: {df['true_severity'].notna().sum()} labeled")

print(f"\nSeverity distribution:")
print(df['true_severity'].value_counts().sort_index())

print(f"\nColumns: {df.columns.tolist()}")
df.head(3)

Loaded 200 labeled reviews

Labels present:
  true_sentiment: 200 labeled
  true_severity: 200 labeled

Severity distribution:
true_severity
1    26
2    18
3    69
4    42
5    45
Name: count, dtype: int64

Columns: ['review_id', 'app_name', 'rating', 'review_text', 'vader_sentiment', 'severity_score', 'severity_label', 'is_hidden_negative', 'true_sentiment', 'true_severity', 'notes', 'notes_new']


,review_id,app_name,rating,review_text,vader_sentiment,severity_score,severity_label,is_hidden_negative,true_sentiment,true_severity,notes,notes_new
0,cf528bb3-494c-465d-8291-c138a7777336,Chime,5,"No fees, so many features. Chime Rocks",Negative,1,Low,False,Positive,1,NaN,NaN
1,f1edde65-f460-469f-bef1-2136ebc74fa8,Cash App,1,hater,Negative,1,Low,False,Negative,5,NaN,NaN
2,68f381c9-606f-4a71-a37d-5e96cfdd28f6,Cash App,3,Very useful app as long as I was sending money...,Positive,5,Critical,False,Negative,4,NaN,NaN


## Check Column Names

Before we analyze, let's confirm what columns exist in the labeled dataset.
We need: `human_sentiment`, `human_severity`, `predicted_sentiment`, `predicted_severity`

In [47]:
print("column names in the labeled dataset")
for col in df.columns:
    print(f" - {col}")

#Now we show data types
print(f"\nData types of the columns:")
print(f"\n{df.dtypes}")
display(df.dtypes)

column names in the labeled dataset
 - review_id
 - app_name
 - rating
 - review_text
 - vader_sentiment
 - severity_score
 - severity_label
 - is_hidden_negative
 - true_sentiment
 - true_severity
 - notes
 - notes_new

Data types of the columns:

review_id                 str
app_name                  str
rating                  int64
review_text               str
vader_sentiment           str
severity_score          int64
severity_label            str
is_hidden_negative       bool
true_sentiment            str
true_severity           int64
notes                 float64
notes_new             float64
dtype: object


review_id                 str
app_name                  str
rating                  int64
review_text               str
vader_sentiment           str
severity_score          int64
severity_label            str
is_hidden_negative       bool
true_sentiment            str
true_severity           int64
notes                 float64
notes_new             float64
dtype: object

# 📊 Severity Agreement Analysis

We compare human-labeled severity (`true_severity`) against model severity
(`severity_score`) to understand how far off the rule-based engine is.


In [48]:
df['severity_match'] = df['true_severity'] == df['severity_score']
df['severity_error'] = abs(df['true_severity'] - df['severity_score'])

print("SEVERITY AGREEMENT")
print("=" * 50)
print(f"Exact match rate: {df['severity_match'].mean():.1%}")
print(f"Mean absolute error: {df['severity_error'].mean():.2f}")
print("\nError distribution:")
print(df['severity_error'].value_counts().sort_index())


SEVERITY AGREEMENT
Exact match rate: 20.0%
Mean absolute error: 1.66

Error distribution:
severity_error
0    40
1    56
2    57
3    27
4    20
Name: count, dtype: int64


# 😊 Sentiment Agreement Analysis

We compare human-labeled sentiment (`true_sentiment`) against VADER sentiment
(`vader_sentiment`) to understand VADER’s weaknesses.


In [49]:
df['sentiment_match'] = df['true_sentiment'] == df['vader_sentiment']

print("SENTIMENT AGREEMENT")
print("=" * 50)
print(f"Exact match rate: {df['sentiment_match'].mean():.1%}")

print("\nConfusion matrix:")
print(pd.crosstab(df['true_sentiment'], df['vader_sentiment'],
                  rownames=['Human'], colnames=['Model']))


SENTIMENT AGREEMENT
Exact match rate: 67.0%

Confusion matrix:
Model     Negative  Neutral  Positive
Human                                
Negative        91       14        32
Neutral          6        4        10
Positive         2        2        39


In [50]:
# Check what columns we have
print("Columns in the dataframe:")
print(df.columns.tolist())

print(f"\nDataframe shape: {df.shape}")
print(f"First few rows:")
print(df.head())

# Check for any columns with 'sentiment' or 'severity' in the name
sentiment_cols = [col for col in df.columns if 'sentiment' in col.lower()]
severity_cols = [col for col in df.columns if 'severity' in col.lower()]

print(f"\nColumns with 'sentiment': {sentiment_cols}")
print(f"Columns with 'severity': {severity_cols}")

Columns in the dataframe:
['review_id', 'app_name', 'rating', 'review_text', 'vader_sentiment', 'severity_score', 'severity_label', 'is_hidden_negative', 'true_sentiment', 'true_severity', 'notes', 'notes_new', 'severity_match', 'severity_error', 'sentiment_match']

Dataframe shape: (200, 15)
First few rows:
                              review_id  app_name  rating  \
0  cf528bb3-494c-465d-8291-c138a7777336     Chime       5   
1  f1edde65-f460-469f-bef1-2136ebc74fa8  Cash App       1   
2  68f381c9-606f-4a71-a37d-5e96cfdd28f6  Cash App       3   
3  93c370ea-133e-4f06-b1ea-d9dc6ef316c1     Venmo       1   
4  20b991b7-1a38-4040-8678-ef627a530f4d    PayPal       1   

                                         review_text vader_sentiment  \
0             No fees, so many features. Chime Rocks        Negative   
1                                              hater        Negative   
2  Very useful app as long as I was sending money...        Positive   
3  I hate the fact that you have to

**Confusion Matrix Interpretation**:
- Rows = what is labeled
- Columns = what MODEL predicted
- Diagonal = correct predictions
- Off-diagonal = errors (e.g., you said Negative, model said Positive)

## 2. High-Severity Misses (Critical Failures)

**This is the most important section.**

We're looking for reviews where:
- I labeled severity 4 or 5 (critical service failure)
- MODEL predicted severity 1-3 (minor or no issue)

These are the dangerous misses - real customer crises the model ignored.

In [51]:
# Find reviews YOU labeled as critical (4-5) but MODEL missed (1-3)
critical_misses = df[
    (df['true_severity'] >= 4) & 
    (df['severity_score'] <= 3)
].copy()

print(f"🚨 CRITICAL MISSES: {len(critical_misses)} reviews")
print("=" * 50)
print("These are service failures you identified that the model completely missed.\n")

# Calculate miss magnitude
critical_misses['miss_magnitude'] = critical_misses['true_severity'] - critical_misses['severity_score']
critical_misses_sorted = critical_misses.sort_values('miss_magnitude', ascending=False)

print(f"Worst miss: You labeled {critical_misses_sorted.iloc[0]['true_severity']}, model predicted {critical_misses_sorted.iloc[0]['severity_score']}")

🚨 CRITICAL MISSES: 44 reviews
These are service failures you identified that the model completely missed.

Worst miss: You labeled 5, model predicted 1


In [52]:
# Find ALL severity disagreements (not just critical misses)
severity_errors = df[df['severity_match'] == False].copy()

print(f"📊 ALL SEVERITY DISAGREEMENTS: {len(severity_errors)} reviews")
print("=" * 50)

# Show error patterns
print("\nError patterns (Human vs Model):")
error_patterns = pd.crosstab(
    severity_errors['true_severity'], 
    severity_errors['severity_score'],
    rownames=['Human'], 
    colnames=['Model']
)
print(error_patterns)

# Find the biggest errors (largest absolute difference)
severity_errors['error_magnitude'] = abs(severity_errors['true_severity'] - severity_errors['severity_score'])
biggest_errors = severity_errors.sort_values('error_magnitude', ascending=False)

print(f"\nBiggest disagreements (top 5):")
for idx, row in biggest_errors.head(5).iterrows():
    print(f"\nHuman: {row['true_severity']} | Model: {row['severity_score']} | Diff: {row['error_magnitude']}")
    print(f"App: {row['app_name']} | Rating: {row['rating']}★")
    print(f"Review: {row['review_text'][:200]}...")
    print("-" * 80)

📊 ALL SEVERITY DISAGREEMENTS: 160 reviews

Error patterns (Human vs Model):
Model   1  2  3   4   5
Human                  
1       0  0  1   4   3
2       7  0  0   5   6
3      30  1  0  19  16
4      16  0  5   0  14
5      17  1  5  10   0

Biggest disagreements (top 5):

Human: 5 | Model: 1 | Diff: 4
App: Cash App | Rating: 1★
Review: hater...
--------------------------------------------------------------------------------

Human: 5 | Model: 1 | Diff: 4
App: PayPal | Rating: 1★
Review: garbage,I feel like an app that's accepted worldwide almost as the app we get free money with using mobile games shouldn't feel like it's completely against you,I have one PayPal account that won't le...
--------------------------------------------------------------------------------

Human: 5 | Model: 1 | Diff: 4
App: Chime | Rating: 1★
Review: Automatically installed itself onto my phone after an update. Shady behavior from a "banking" app. I wouldn't trust them with a dollar....
-----------------

In [53]:
# Load and inspect the labels.csv file
labels_file = Path(r"C:\Users\kirim\Downloads\labels.csv")
df_test = pd.read_csv(labels_file)

print("Columns in labels.csv:")
print(df_test.columns.tolist())

print("\nFirst 3 rows:")
print(df_test.head(3))

print("\nColumn data types:")
print(df_test.dtypes)

# Check if there are ANY non-null values in columns with 'sentiment' or 'severity'
for col in df_test.columns:
    if 'sentiment' in col.lower() or 'severity' in col.lower():
        non_null = df_test[col].notna().sum()
        print(f"\n{col}: {non_null} non-null values")
        if non_null > 0:
            print(f"  Sample values: {df_test[col].dropna().head(3).tolist()}")

Columns in labels.csv:
['review_id', 'true_sentiment', 'true_severity', 'notes']

First 3 rows:
                              review_id true_sentiment  true_severity  notes
0  cf528bb3-494c-465d-8291-c138a7777336       Positive              1    NaN
1  f1edde65-f460-469f-bef1-2136ebc74fa8       Negative              5    NaN
2  68f381c9-606f-4a71-a37d-5e96cfdd28f6       Negative              4    NaN

Column data types:
review_id             str
true_sentiment        str
true_severity       int64
notes             float64
dtype: object

true_sentiment: 200 non-null values
  Sample values: ['Positive', 'Negative', 'Negative']

true_severity: 200 non-null values
  Sample values: [1, 5, 4]


## 3. Pattern Detection in Misses

Let's extract the most common words appearing in reviews the model got wrong.
This will help us identify what crisis language we need to add to the lexicon.

In [54]:
from collections import Counter
import re

def extract_keywords(text):
    """Extract meaningful keywords from review text"""
    text = text.lower()
    # Remove punctuation, split into words
    words = re.findall(r'\b[a-z]{3,}\b', text)
    # Filter out common stopwords
    stopwords = {'the', 'and', 'this', 'that', 'with', 'for', 'was', 'but', 
                 'have', 'not', 'are', 'you', 'can', 'they', 'been', 'your',
                 'from', 'all', 'use', 'app', 'just', 'like', 'get'}
    return [w for w in words if w not in stopwords]

# Analyze severity errors
severity_errors = df[df['severity_match'] == False].copy()
error_text = ' '.join(severity_errors['review_text'].astype(str))
error_keywords = extract_keywords(error_text)
error_freq = Counter(error_keywords).most_common(30)

print("🔍 TOP KEYWORDS IN SEVERITY ERRORS")
print("=" * 50)
print("These words appear frequently in reviews where severity was misclassified:\n")
for word, count in error_freq:
    print(f"  {word:15s} : {count:3d} times")

🔍 TOP KEYWORDS IN SEVERITY ERRORS
These words appear frequently in reviews where severity was misclassified:

  money           :  71 times
  account         :  61 times
  card            :  35 times
  chime           :  32 times
  when            :  26 times
  cash            :  25 times
  had             :  23 times
  paypal          :  22 times
  has             :  21 times
  out             :  20 times
  service         :  20 times
  now             :  20 times
  would           :  20 times
  bank            :  19 times
  even            :  19 times
  will            :  19 times
  used            :  17 times
  because         :  17 times
  don             :  17 times
  one             :  16 times
  help            :  16 times
  them            :  16 times
  never           :  16 times
  still           :  16 times
  time            :  15 times
  then            :  15 times
  customer        :  15 times
  using           :  14 times
  there           :  14 times
  new             : 

**Action**: These keywords should be added to our fintech crisis lexicon.
Words like "money", "account", "transfer", "frozen", "locked" are strong signals of severity issues.

## 4. Negation Detection Gaps

Negation words (not, can't, won't, never) flip sentiment meaning.
Example: "not good" should be NEGATIVE, but VADER might read "good" → POSITIVE.

Let's find reviews with negation where sentiment was misclassified.

In [55]:
# Define negation patterns
negation_patterns = [
    r'\bnot\b', r'\bno\b', r'\bnever\b', r'\bcan\'t\b', r'\bcannot\b',
    r'\bwon\'t\b', r'\bwouldn\'t\b', r'\bdidn\'t\b', r'\bdoesn\'t\b',
    r'\bnothing\b', r'\bnobody\b', r'\bnowhere\b', r'\bdon\'t\b'
]

df['has_negation'] = df['review_text'].str.contains(
    '|'.join(negation_patterns), case=False, regex=True, na=False
)

negation_mismatches = df[
    df['has_negation'] & 
    (df['sentiment_match'] == False)
]

print(f"🔄 NEGATION-RELATED SENTIMENT ERRORS: {len(negation_mismatches)} reviews")
print("=" * 50)
print(f"Out of {df['has_negation'].sum()} reviews with negation, {len(negation_mismatches)} had sentiment errors")
print(f"Error rate: {len(negation_mismatches) / df['has_negation'].sum() * 100:.1f}%\n")

🔄 NEGATION-RELATED SENTIMENT ERRORS: 43 reviews
Out of 112 reviews with negation, 43 had sentiment errors
Error rate: 38.4%



In [56]:
# Show examples of negation errors
print("Examples of negation-related sentiment errors:\n")
for idx, row in negation_mismatches.head(8).iterrows():
    print(f"Human: {row['true_sentiment']:8s} | Model: {row['vader_sentiment']:8s}")
    print(f"Review: {row['review_text'][:250]}...")
    print("-" * 80)

Examples of negation-related sentiment errors:

Human: Positive | Model: Negative
Review: No fees, so many features. Chime Rocks...
--------------------------------------------------------------------------------
Human: Negative | Model: Positive
Review: garbage,I feel like an app that's accepted worldwide almost as the app we get free money with using mobile games shouldn't feel like it's completely against you,I have one PayPal account that won't let me order a physical card no matter what I do,my ...
--------------------------------------------------------------------------------
Human: Negative | Model: Positive
Review: Usually didn't require help but when needed help is hard to get...
--------------------------------------------------------------------------------
Human: Negative | Model: Neutral 
Review: Chime did not explain anything to me at all and told me that I couldn't do it....
--------------------------------------------------------------------------------
Human: Negative

**Action**: We need to add negation handling BEFORE VADER scoring.
This will flip the polarity when negation words appear near sentiment words.

## 5. Fintech Crisis Language

Check for financial crisis keywords that should automatically trigger high severity.
Things like "account frozen", "can't access money", "lost funds", etc.

In [57]:
# Define financial crisis keywords
crisis_keywords = [
    r'\block', r'\bfreeze', r'\bfrozen', r'\bheld\b', r'\bsuspend', r'\bblock', r'\brestrict',
    r'can\'t access', r'won\'t load', r'wouldn\'t let', r'can\'t withdraw',
    r'\bdeclined\b', r'\bdenied\b', r'\breject', r'\bfraud\b', r'\bscam', r'\bstolen',
    r'lost money', r'\bmissing\b', r'disappeared', r'wrong account',
    r'\bhacked\b', r'\bdispute', r'\bhold\b', r'\bweeks\b',
    r'\brent\b', r'\bbills\b', r'\bemergency', r'\burgent', r'\bdesperate'
]

df['has_crisis_keyword'] = df['review_text'].str.contains(
    '|'.join(crisis_keywords), case=False, regex=True, na=False
)

# Find high-severity reviews (you labeled 4-5) with crisis keywords that model underestimated
crisis_underestimated = df[
    df['has_crisis_keyword'] & 
    (df['true_severity'] >= 4) & 
    (df['severity_score'] <= 3)
]

print(f"💰 FINANCIAL CRISIS LANGUAGE ANALYSIS")
print("=" * 50)
print(f"Total reviews with crisis keywords: {df['has_crisis_keyword'].sum()}")
print(f"Reviews you labeled high-severity (4-5) with crisis keywords: {len(df[(df['has_crisis_keyword']) & (df['true_severity'] >= 4)])}")
print(f"Of those, model underestimated (predicted 1-3): {len(crisis_underestimated)}")

💰 FINANCIAL CRISIS LANGUAGE ANALYSIS
Total reviews with crisis keywords: 75
Reviews you labeled high-severity (4-5) with crisis keywords: 39
Of those, model underestimated (predicted 1-3): 9


In [58]:
# Show examples
print("\nExamples of crisis language the model underestimated:\n")
for idx, row in crisis_underestimated.head(6).iterrows():
    print(f"App: {row['app_name']} | Rating: {row['rating']}★")
    print(f"Human: Severity {row['true_severity']} | Model: Severity {row['severity_score']}")
    print(f"Review: {row['review_text'][:300]}...")
    print("-" * 80)


Examples of crisis language the model underestimated:

App: Venmo | Rating: 1★
Human: Severity 5 | Model: Severity 1
Review: Adding money to your account, they take it out of your bank account the next day and then keep your money for 10 days after the transfer. cash app is much better. your funds are available right away and they don't hold your .money for over a week!...
--------------------------------------------------------------------------------
App: Chime | Rating: 3★
Human: Severity 4 | Model: Severity 1
Review: Chime have you started to forget the people that made you #1. The🥇that still hold strong are slipping away, leaving a bad taste with long time users. Yet still have verity of Options, few Brick banks & online can match🥈Secured Visa card really builds credit & works almost anywhere🥇online check cashi...
--------------------------------------------------------------------------------
App: PayPal | Rating: 1★
Human: Severity 5 | Model: Severity 1
Review: The worst pay se

## 6. Actionable Improvements Summary

Based on the error analysis, here are the prioritized improvements we'll implement in `src/analysis.py`.

In [59]:
print("\n" + "=" * 80)
print("📋 IMPROVEMENT PRIORITIES FOR src/analysis.py")
print("=" * 80)

severity_errors = df[df['severity_match'] == False]
sentiment_errors = df[df['sentiment_match'] == False]

print(f"""
1. NEGATION HANDLING
   - Found {len(negation_mismatches)} sentiment errors involving negation
   - Error rate on negated reviews: {len(negation_mismatches) / df['has_negation'].sum() * 100:.1f}%
   - Priority: HIGH
   - Action: Pre-process negations before VADER scoring
   
2. FINTECH CRISIS LEXICON
   - Found {len(crisis_underestimated)} high-severity misses with crisis keywords
   - Priority: HIGH
   - Action: Add domain-specific severity boosters for money/access issues
   
3. TOP MISSED KEYWORDS
   - Most common in errors: {', '.join([w for w, c in error_freq[:10]])}
   - Priority: MEDIUM
   - Action: Add these to severity detection rules
   
4. OVERALL SEVERITY CALIBRATION
   - Current exact match: {df['severity_match'].mean() * 100:.1f}%
   - Within ±1 level: {(df['severity_error'] <= 1).sum() / len(df) * 100:.1f}%
   - Mean error: {df['severity_error'].mean():.2f} severity points
   - Priority: HIGH
   - Action: Recalibrate severity thresholds, add rating-based fallback
   
5. SENTIMENT ACCURACY
   - Current accuracy: {df['sentiment_match'].mean() * 100:.1f}%
   - Target: >75% after negation handling
""")

print("=" * 80)
print("✅ Error analysis complete!")
print("=" * 80)
print("\nNext step: Update src/analysis.py with these improvements")


📋 IMPROVEMENT PRIORITIES FOR src/analysis.py

1. NEGATION HANDLING
   - Found 43 sentiment errors involving negation
   - Error rate on negated reviews: 38.4%
   - Priority: HIGH
   - Action: Pre-process negations before VADER scoring

2. FINTECH CRISIS LEXICON
   - Found 9 high-severity misses with crisis keywords
   - Priority: HIGH
   - Action: Add domain-specific severity boosters for money/access issues

3. TOP MISSED KEYWORDS
   - Most common in errors: money, account, card, chime, when, cash, had, paypal, has, out
   - Priority: MEDIUM
   - Action: Add these to severity detection rules

4. OVERALL SEVERITY CALIBRATION
   - Current exact match: 20.0%
   - Within ±1 level: 48.0%
   - Mean error: 1.66 severity points
   - Priority: HIGH
   - Action: Recalibrate severity thresholds, add rating-based fallback

5. SENTIMENT ACCURACY
   - Current accuracy: 67.0%
   - Target: >75% after negation handling

✅ Error analysis complete!

Next step: Update src/analysis.py with these improvem